# Sentinel-2 L2A NDVI (Element84)

Companion to `scripts/element84_s2.py`: 100 km2 around Paris, cloud cover <= 5 %, then mapped with `eo_stac.viz`.

Needs `jupyter`, e.g. `uv run --with jupyter jupyter lab`.

In [ ]:
import math
from pathlib import Path

import rioxarray
import xarray as xr

from eo_stac import Sentinel2Params, STACSource, download_items, search_items, viz

In [ ]:
S2_OUT = Path("../data/s2_paris")
lon, lat, half_km = 2.3522, 48.8566, 5.0
dlat = half_km / 111.32
dlon = half_km / (111.32 * math.cos(math.radians(lat)))
S2_BBOX = [lon - dlon, lat - dlat, lon + dlon, lat + dlat]

s2 = search_items(
    STACSource.from_url("https://earth-search.aws.element84.com/v1"),
    collections=["sentinel-2-l2a"],
    bbox=S2_BBOX,
    datetime="2026-08-01/2026-08-15",
    params=Sentinel2Params(max_cloud_cover=5),
)
print(len(s2), "items")
s2 = download_items(s2, S2_OUT, assets=["red", "green", "blue", "nir"], bbox=S2_BBOX)
s2[["item_id", "datetime", "cloud_cover"]]

In [ ]:
viz.plot_footprints(s2, column="cloud_cover")

In [ ]:
viz.plot_thumbnails(s2, max_items=6)

### NDVI with rioxarray

`NDVI = (NIR - Red) / (NIR + Red)`. L2A scenes processed with baseline 04.00 or later
(since 2022) carry a +1000 offset, removed here before the ratio.

In [ ]:
def reflectance(path):
    da = rioxarray.open_rasterio(path, masked=True).squeeze("band", drop=True)
    return ((da - 1000) / 10000).clip(0, 1)


def band_file(item_dir, asset):
    return next(item_dir.glob(f"*_{asset}_sub-*.tif"))


item_dirs = sorted({p.parent for p in S2_OUT.rglob("*_red_sub-*.tif")})
ndvi_by_item = {}
for d in item_dirs:
    red, nir = reflectance(band_file(d, "red")), reflectance(band_file(d, "nir"))
    ndvi_by_item[d.name] = ((nir - red) / (nir + red)).rename("ndvi")

for name, ndvi in ndvi_by_item.items():
    print(
        f"{name}: mean {float(ndvi.mean()):.3f}, "
        f"vegetated (>0.4) {float((ndvi > 0.4).mean()):.0%}"
    )

In [ ]:
last_dir = item_dirs[-1]
ndvi = ndvi_by_item[last_dir.name]
ndvi_path = S2_OUT / f"{last_dir.name}_ndvi.tif"
ndvi.rio.write_crs(ndvi.rio.crs).rio.to_raster(ndvi_path)
viz.quick_view(ndvi_path, overview_level=None)

### True colour and NIR-red-green false colour

In [ ]:
def stretched(path):
    return (reflectance(path) / 0.3).clip(0, 1)


def save_composite(assets, name):
    bands = [stretched(band_file(last_dir, a)) for a in assets]
    rgb = xr.concat(bands, dim="band").assign_coords(band=[1, 2, 3])
    rgb = (rgb.fillna(0) * 255).astype("uint8").rio.write_crs(bands[0].rio.crs)
    path = S2_OUT / f"{last_dir.name}_{name}.tif"
    rgb.rio.to_raster(path)
    return path


viz.quick_view(
    save_composite(["red", "green", "blue"], "rgb"),
    band_indices=[1, 2, 3],
    overview_level=None,
)

In [ ]:
# Vegetation shows bright red in NIR-red-green
viz.quick_view(
    save_composite(["nir", "red", "green"], "nrg"),
    band_indices=[1, 2, 3],
    overview_level=None,
)